# Propagation of Hollow Core waveguide at low loss wavelength

Propagating input fields in a device capable of resonant behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import slab2 as sg
from scipy.optimize import newton

plt.style.use('dark_background')
%matplotlib widget

# Setup Waveguide and input fields

In [ ]:
loss_min_wl = 5.828070175438596e-06
max_loss_wl = 6.90764472e-06

In [ ]:
A = sg.SlabGuide(scale=1e-5, Ts=[6.5, 1, 2, 1, 6.5],  ns=[1, 1.44, 1, 1.44, 1],
                 xrefs=[100, 40, 300, 40, 100], wl=loss_min_wl)
A.Offsets = np.zeros_like(A.Offsets)
A.model_kwargs['Offsets'] = A.Offsets

In [ ]:
half_width = 6
shift = .5
amplitude = 1

Lx = -half_width + shift
Rx = half_width + shift

def quadratic_piecewise(x):
    x = np.array(x, dtype=float)
    conds = [(x < Lx), ((Lx <= x)*(x <= Rx)), (x > Rx)]
    funcs = [lambda x: 0, lambda x: amplitude * (1 - ((x - shift) / half_width)**2), lambda x: 0]
    return np.piecewise(x, conds, funcs)

def quadratic_piecewise2d(x, zs=None, wavenumber=A.K0):
    fx = quadratic_piecewise(x)
    fz = np.exp(1j * wavenumber * zs)
    return np.outer(fz, fx)

f0 = quadratic_piecewise
F0 = quadratic_piecewise2d

In [ ]:
A.plot_refractive_index();
A.add_1d_plot(f0, label='$f_0$')

# Debugging Guided Mode coefficients

These are very hard to get using the standard transfer matrix method.  Can we do better somehow?

In [ ]:
A.eigenvalue_plot(-8, 8, -.4, A.Z_max_guided.imag+2);

In [ ]:
A.get_guided_propagation_constants()
sg.plot_points(A.guided_mode_propagation_constants)

In [ ]:
Z = A.guided_mode_propagation_constants[0]
Z

In [ ]:
def M(x, Z, i, **kwargs):
    return np.cosh(1j * Z * x)

def N(x, Z, i, **kwargs):
    return np.sinh(1j * Z * x)
    
def Mp(x, Z, i, **kwargs):
    return 1j * Z * N(x, Z, i, **kwargs)

def Np(x, Z, i, **kwargs):
    return 1j * Z * M(x, Z, i, **kwargs)
    
A.regional_models[2] = {'M':M, 'N':N, 'dMdx':Mp, 'dNdx':Np}

In [ ]:
sg.eigenvalue_plot(A, -8, 8, -.4, A.Z_max_guided.imag+2);

In [ ]:
A.transfer_matrix(Z,1)

In [ ]:
A.get_guided_modes()

In [ ]:
sg.plot_field_1d(A, A.guided_modes[0])

For non-toy example case, none of the guided modes significantly overlap the input function at non-resonant input frequencies.

# Propagate Radiation Field

In [ ]:
N = sg.OurMethod
P = sg.Propagator(A, f0=f0, Lx=Lx, Rx=Rx, Normalization_Class=N)

### Propagating Portion

This part takes a lot of resolution in this case

#### Spike 1

In [ ]:
Np = 50
Lz = .001
Rz = .385

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
P.plot_transform();

In [ ]:
Np = 120
Lz = .385
Rz = .395

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
P.plot_transform([1]);

#### Spike 2

In [ ]:
Np = 60
Lz = .395
Rz = 1.14

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
plt.close('all')
P.plot_transform(indices=[2])

In [ ]:
Np = 150
Lz = 1.14
Rz = 1.22

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
plt.close('all')
P.plot_transform(indices=[3])

#### Spike 3

In [ ]:
Np = 80
Lz = 1.22
Rz = 1.85

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
plt.close('all')
P.plot_transform(indices=[3,4])

In [ ]:
Np = 80
Lz = 1.85
Rz = 2.05

P.add_contour(sg.real_contour(Lz, Rz, Np))

#### Spike 4

In [ ]:
Np = 60
Lz = 2.05
Rz = 2.65

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
Np = 50
Lz = 2.65
Rz = 2.8

P.add_contour(sg.real_contour(Lz, Rz, Np))

#### Remaining propagating portion

In [ ]:
Np = 100
Lz = 2.8
Rz = 5.1

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
Np = 75
Lz = 5.1
Rz = A.Z_evanescent

P.add_contour(sg.real_contour(Lz, Rz, Np))

### Evanescent Portion

In [ ]:
Np = 70
Lz = A.Z_evanescent
Rz = 16.5

P.add_contour(sg.real_contour(Lz, Rz, Np))

In [ ]:
Np = 50
Lz = 16.5
Rz = 22

P.add_contour(sg.real_contour(Lz, Rz, Np))

## Full transform Plot

In [ ]:
plt.close('all')

P.plot_transform();

# Cross sections at input


In [ ]:
sg.plot_field_1d(A, P, label='estimate') 
sg.add_1d_plot(A, f0, label='true $f_0$')
# plt.gca().set_xlim(-3,3)

In [ ]:
sg.plot_field_1d(A, lambda x, z: f0(x) - P(x,z), label='difference') 
sg.add_1d_plot(A, f0, label='true $f_0$');


# 2D Filled Contour Plots

In [ ]:
# Good ratio for z is 6 points per unit
zmin, zmax=0, 60
sg.plot_field_2d(A, P, zmin=zmin,
                zmax=zmax, zref=int((zmax-zmin)*6),
               );

# Surface Plot

In [ ]:
# zmin, zmax=0, 50
# A.plot_field_2d_surface(Full, xs=B.all_Xs, colorbar=False,
#                 azim=-60, elev=30, zoom=3,
#                 zmin=0, zmax=zmax, zref=int(zmax*20),
#                 rstride=1, cstride=1,
#                );

# Leaky mode representation via contour integration


In [ ]:
fig, ax, _, _ = A.determinant_plot(0, 1.3*A.Z_evanescent, -1, 1, iref=250, rref=250, levels=70,
                                  Normalizer=Normalizer);
plt.vlines(A.Z_evanescent, *ax.get_ylim(), lw=1.2, ls='--', color='k');

In [ ]:
z_guesses = np.array([.4, 1.1, 1.9, 2.8, 3.5, 4.2,
                      4.98 - .125j, 5.6 - .25j, 6.15 - .2j, 6.8 - 1.2j,
                     7.5 - .1j, 8.25 - .85j, 9 - .1j, 9.75 - .13j,
                     ])
Zs = []
for z in z_guesses:
    Zs.append(newton(A.determinant, z, args=('Z', 'guided', 'TE', Normalizer)))
Zs = np.array(Zs)

for Z in Zs:
    plt.plot(Z.real, Z.imag, marker='o')

## Construct field

In [ ]:
P_leaky = []
paper_method=False

for Z in Zs:
    PZe = Propagator_Class(A.circular_contour(Z, 1e-6, 5, 'cw'), f0=f0,
                           Lx=-b, Rx=b, field_type=field_type, 
                           sign='+1', paper_method=paper_method)
    P_leaky.append(PZe)
    
    # PZo = Propagator_Class(A.circular_contour(Z, 1e-6, 5),
    #                        f0=f0, Lx=-b, Rx=b,
    #                        field_type=field_type,
    #                        sign='-1',
    #                        paper_method=paper_method)
    # P_leaky.append(PZo)

In [ ]:
full_leaky = lambda x : sum([P.slice_propagate(x) for P in P_leaky])
Full_leaky = lambda x, z : sum([P.propagate(x, zs=z) for P in P_leaky])
Diff = lambda x,z : Full_leaky(x,z) - Full(x,z)

## Leaky field profile at input

In [ ]:
A.plot_field_1d(full_leaky, label='leaky field')
A.add_1d_plot(f0, label='$f_0$')

## 2D overhead plot of leaky mode field approximation

In [ ]:
plt.close('all')
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharey=True, figsize=(11,8))

xs = B.all_Xs

zmin = 0
zmax = 100
zref = int((zmax-zmin)*40/2)
zs = np.linspace(zmin, zmax, zref)

In [ ]:
A.add_2d_plot(Full, ax=ax1, xs=xs, zs=zs, plot_Rhos=True);
ax1.set_title('True Field')

In [ ]:
A.add_2d_plot(Full_leaky, ax=ax2, xs=xs, zs=zs, plot_Rhos=True);
ax2.set_title('Leaky Mode Field')

In [ ]:
A.add_2d_plot(Diff, ax=ax3, xs=xs, zs=zs, part='norm', plot_Rhos=True);
ax3.set_title('Norm of Difference')

## Asymptotics (slice along z-axis at x=0)

In [ ]:
slice_at_x = 0
constant_variable = 'x'

full_z_leaky_ours = lambda z: sum([P.slice_propagate(z, slice_at=slice_at_x, constant_variable=constant_variable) for P in P_leaky])
full_z = lambda z: sum([f.slice_propagate(z, slice_at=slice_at_x, constant_variable=constant_variable) for f in propfuncs]) 

ratio = lambda z: full_z_leaky_ours(z) / full_z(z)


In [ ]:
part = 'real'
zmin, zmax = 0, 100
zref=int((zmax-zmin)*40)
zs = np.linspace(zmin, zmax, zref)

A.plot_field_1d(full_z, xs=zs, part=part, plot_axis=False, plot_Rhos=False, linewidth=2, label='true');
A.add_1d_plot(full_z_leaky_ours, xs=zs, part=part, linewidth=1, linestyle='--', label='leaky');
# A.add_1d_plot(ratio, xs=zs, part=part, linewidth=1, linestyle='--', label='ratio');


In [ ]:
part = 'real'
zmin, zmax = 1000, 1200
zref=int((zmax-zmin)*40)
zs = np.linspace(zmin, zmax, zref)

A.plot_field_1d(full_z, xs=zs, part=part, plot_axis=False, plot_Rhos=False, linewidth=2, label='true');
A.add_1d_plot(full_z_leaky_ours, xs=zs, part=part, linewidth=1, linestyle='--', label='leaky');
# A.add_1d_plot(ratio, xs=zs, part=part, linewidth=1, linestyle='--', label='ratio');


# Integral along Steepest Descent Path in Z plane

Here we examine the remaining portion of the expansion, namely the integral along the steepest descent path (SDP).

The SDP is found by keeping the imaginary part of $\beta(Z) = i \sqrt{k_0^2n_0^2 - Z^2}$ constant at $\text{Im}(\beta(0)) = k_0 n_0$.  Using $Z = x + y i$ we calculate from the complex square root formula that 

$$ y = \pm \frac{k_0 n_0 x}{\sqrt{k_0^2n_0^2 + x^2}}$$

along this path. Below we plot this over the determinant function.

In [ ]:
def sdp(x, guide=A):
    K = guide.K0 * guide.n0
    return -(K * x) / np.sqrt(K**2 + x**2)

In [ ]:
xmin, xmax= -5, .5
ymin, ymax = -1, 2
xs_sdp = np.linspace(xmin, xmax, 1000)
ys_sdp = sdp(xs_sdp)
msk = np.where((ys_sdp<ymax)*(ys_sdp>ymin))

In [ ]:
plotlogf(A.determinant_Z, xmin, xmax, ymin, ymax, 'TE', 'leaky', iref=200, rref=200, levels=50)
ax = plt.gca()
ax.plot(xs_sdp[msk], ys_sdp[msk]);
# ax.plot(xs[msk], -ys[msk]);

To integrate along this path, we need it's complex derivative.  Using the parameterization

$$Z(x) = x + i y(x) = x \pm i \frac{k_0 n_0 x}{\sqrt{k_0^2n_0^2 + x^2}},$$

from above, we see that the derivative is

$$ \frac{dZ}{dx} = 1 \pm i \left( \frac{k_0 n_0}{\sqrt{k_0^2n_0^2 + x^2}}\right)^3$$

## Propagate along SDP

We're building in the capabilities to do this

In [ ]:
Propagator_Class = A.propagator()

In [ ]:
paper_method = False

In [ ]:
P_sdp1 = Propagator_Class(A.sdp_contour(-.01, -.7, 100), f0=f0, Lx=-b, Rx=b, 
                     field_type=field_type, paper_method=paper_method,
                     sign='+1')
P_sdp2 = Propagator_Class(A.sdp_contour(-.7, -2.5, 100), f0=f0, Lx=-b, Rx=b, 
                     field_type=field_type, paper_method=paper_method,
                     sign='+1')

In [ ]:
fig, ax = P_sdp1.plot_transform(xs=P_sdp1.Zs.real)
P_sdp2.plot_transform(ax=ax, xs=P_sdp2.Zs.real)

In [ ]:
x_slice = 4.

sdp = lambda x: P_sdp1.slice_propagate(x,0) + P_sdp2.slice_propagate(x,0)
sdp_z = lambda z: P_sdp1.slice_propagate(z, x_slice, 'x') + P_sdp2.slice_propagate(z, x_slice, 'x')

Sdp = lambda x, z: P_sdp1.propagate(x, z) + P_sdp2.propagate(x, z)

In [ ]:
A.plot_field_1d(sdp, xs=np.linspace(-6,6,1000), part='imag', plot_Rhos=False)

In [ ]:
A.plot_field_1d(sdp_z, xs=np.linspace(0,40,1400), plot_axis=False, plot_Rhos=False)

In [ ]:
A.plot_field_2d(Sdp, xs=np.linspace(-5,5,1000), zmax=15, zref=1000)

In [ ]:
A.plot_field_2d_surface(P_sdp1.propagate, xs=np.linspace(0, 5, 100), zmax=5, zref=200, cstride=1, rstride=1,
                       colorbar=True)

## Observations

- Seems that space wave is not very large in magnitude in heterogeneous regions.  Does blow up outside of that.
  
- Because of this, likely that space wave is not very important for asymptotic approximation in this case.
  
- This might not hold at other wavelengths however, in particular for large wavelengths of this waveguide there are no leaky modes between real axis as SDP, so all asymptotics would be done using space wave.

# Save data for visualization


In [ ]:
xs = A.all_Xs
zs = np.linspace(0, 100, int(100*40/2))

In [ ]:
f0s = f0(xs)

In [ ]:
rad_ys_propagating = Prop(xs, zs)


In [ ]:
rad_ys_evanescent = Ev(xs, zs)


In [ ]:
# guided_ys = alpha * Fs1(xs, zs=zs)

In [ ]:
Xs, Zs = np.meshgrid(xs, zs)

In [ ]:
def get_trig_indices(mesh_array):
    '''Get triangle index list for three js mesh formation via indexed buffer geometry.'''
    if len(mesh_array.shape) <= 1:
        raise ValueError('Provide meshgrid array as input.')
    Nz, Nx = mesh_array.shape[0], mesh_array.shape[1]
    l_inds = Nx * Nz
    indices =  np.arange(0, l_inds).reshape((Nx, Nz)).T  # Note transpose
    fbases = indices[:Nz-1, :Nx-1].T.flatten()  # Note transpose and index switch due to transpose
    f1 = np.array([[i, i+Nz, i+1] for i in fbases], dtype=int)
    f2 = np.array([[i+1, i+Nz, i+Nz+1] for i in fbases], dtype=int)
    return np.concatenate((f1, f2))


def get_base_vertices(Xs, Zs):
    '''Get vertices of triangles with zero height (y component in three.js).'''
    if Xs.shape != Zs.shape:
        raise ValueError('Provide meshgrid arrays (ie Xs, Zs = np.meshgrid(xs,zs)) as inputs.')
    Base = np.array([Xs, np.zeros_like(Xs), Zs], dtype=float)
    L = Xs.shape[0] * Xs.shape[1]
    base_vertices = Base.T.reshape((L,3))
    return base_vertices


def create_height_array_from_complex(ys, Xs):
    '''Get data for height from complex ys input (needs to be formatted as list:
        [ys[0].real, ys[0].imag, ys[1].real, ys[1].imag, ... ].'''
    L = Xs.shape[0] * Xs.shape[1]
    Ys = np.array([ys.real, ys.imag], dtype=float)
    heights = Ys.T.reshape((L,2))
    return heights


def save_trig_indices_as_json(filepath, mesh_array):
    indices = get_trig_indices(mesh_array)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(indices.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)


def save_base_vertices_as_json(filepath, Xs, Zs):
    base_vertices = get_base_vertices(Xs, Zs)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(base_vertices.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)
        

def save_heights_as_json(filepath, ys, Xs):
    heights= create_height_array_from_complex(ys, Xs)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(heights.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)

def save_values_as_json(filepath, f0s):
    with open(filepath + '.json', 'w') as outfile:
        json.dump(f0s.tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)


In [ ]:
prefix='/home/pv/local/slab/apps/hollow_core/'

In [ ]:
save_trig_indices_as_json(prefix + 'trig_indices_new', Xs)

In [ ]:
save_base_vertices_as_json(prefix + 'base_vertices_new', Xs, Zs)

In [ ]:
# save_heights_as_json(prefix + 'guided_heights', guided_ys, Xs)

In [ ]:
save_heights_as_json(prefix + 'evanescent_heights', rad_ys_evanescent, Xs)

In [ ]:
save_heights_as_json(prefix + 'propagating_heights', rad_ys_propagating, Xs)

In [ ]:
save_values_as_json(prefix + 'f0s', f0s)

In [ ]:
save_values_as_json(prefix + 'x0s', xs)

# Previous leaky mode work

In [ ]:
mode_type = 'leaky'
k_low = A.K_low

In [ ]:
plotlogf(A.determinant, .94*A.K_low, 1.0001*A.K_low, -.1, .1, field_type, mode_type, 
         figsize=(10,5),iref=400, rref=400, levels=200)

### Only even modes necessary

In [ ]:
beta1 = newton(A.determinant, 10.775, args=(field_type, mode_type)).conj()
beta2 = newton(A.determinant, 10.715, args=(field_type, mode_type))
beta3 = newton(A.determinant, 10.6, args=(field_type, mode_type))
beta4 = newton(A.determinant, 10.42, args=(field_type, mode_type))
beta5 = newton(A.determinant, 10.2, args=(field_type, mode_type))

betas = [beta1, beta2, beta3, beta4, beta5]
for beta in betas:
    plt.plot(beta.real, beta.imag, marker='o')


In [ ]:
Fs1 = A.fields(beta1, field_type=field_type, mode_type=mode_type, c0=-1)
Fs2 = A.fields(beta2, field_type=field_type, mode_type=mode_type, c0=-1)
Fs3 = A.fields(beta3, field_type=field_type, mode_type=mode_type, c0=-1)
Fs4 = A.fields(beta4, field_type=field_type, mode_type=mode_type, c0=-1)
Fs5 = A.fields(beta5, field_type=field_type, mode_type=mode_type, c0=-1)

Fs = [Fs1, Fs2, Fs3, Fs4, Fs5]

### Plotting leaky modes

#### 1D plots

In [ ]:
A.plot_field_1d(Fs1)
A.add_1d_plot(Fs2)
A.add_1d_plot(Fs3)
A.add_1d_plot(Fs4)
A.add_1d_plot(Fs5)


### Build "full" leaky field

At this point this is just to get an idea of combining leaky modes.  Scaling should be done via contour integration.

In [ ]:
from scipy.integrate import quad
cfs = np.zeros(len(Fs))
alphas = np.zeros(len(Fs))
for i,F in enumerate(Fs):
    integrand = lambda x: F(x) * F(x).conjugate()
    cfs[i] = quad(integrand, A.Rhos[2], A.Rhos[3], complex_func=True)[0].real
    integrand = lambda x: f0(x) * F(x).conjugate()
    alphas[i] = quad(integrand, A.Rhos[2], A.Rhos[3], complex_func=True)[0].real

# Note: Not clear that this is correct process for scaling leaky fields

In [ ]:
Full_leaky = lambda x, z: sum([Fs[i](x,z) * alphas[i]/cfs[i] for i in range(len(Fs))])

### 2D plots

In [ ]:
xs = np.linspace(-20,20,1000)

In [ ]:
fig, ax = A.plot_field_2d(Full_leaky, xs=xs, zmin=0, zmax=100, zref=2000, colorbar=False)
plt.axis('equal')